# Context-Aware Multimodal Knowledge Retrieval System
## Complete End-to-End RAG Pipeline for PDF / Research Paper Analysis

**What this notebook does:**
- Extracts **all content** from any PDF: text, tables, images, charts, figures, equations
- Summarizes each modality using the best LLM for that content type:
  - **Groq / Llama-3.1** → fast, free text & table summarization
  - **Gemini 1.5 Flash (Vision)** → image, chart, figure, diagram understanding
- Embeds summaries using **HuggingFace sentence-transformers**
- Stores in **ChromaDB** with a **Multi-Vector Retrieval** strategy (summary → original)
- On any query: retrieves relevant text + tables + images and generates a cited answer via **Gemini**
- Shows all **source elements** inline: rendered tables (HTML), inline images, text with page refs

---
### Architecture
```
PDF ──► Unstructured Parser ──► Text / Tables / Images
            │                        │
            │         ┌──────────────┤──────────────────────┐
            │         ▼              ▼                       ▼
            │    Groq/Llama     Groq/Llama           Gemini Vision
            │    Text Summary   Table Summary         Image Summary
            │         │              │                       │
            │         └──────────────┴───────────────────────┘
            │                        │
            │              HuggingFace Embeddings
            │                        │
            │                   ChromaDB
            │           (summaries → original docs)
            │                        │
            └──────────► MultiVectorRetriever
                                     │
                              User Question
                                     │
                           Gemini 1.5 Flash
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
             Answer (Cited)              Sources (Text/Table/Image)
```

## Step 1 — Install & Verify Dependencies
All packages are pre-installed in the `multimodal_env` virtual environment.
Run the cell below only if you need to install/update packages.

In [ ]:
# ── Optional: Install / update packages ────────────────────────────────────────
# Uncomment and run if any import fails below

# import subprocess, sys
# pkgs = [
#     "python-dotenv", "unstructured[all-docs]", "pdfplumber", "pymupdf", "pillow", "lxml",
#     "langchain>=0.3", "langchain-core>=0.3", "langchain-community>=0.3",
#     "langchain-google-genai>=2.0", "langchain-groq>=0.2", "langchain-huggingface>=0.1",
#     "google-generativeai>=0.8", "sentence-transformers>=3.0", "chromadb>=0.5",
#     "tiktoken", "pandas", "numpy", "opencv-python-headless", "matplotlib",
#     "ipywidgets", "tqdm",
# ]
# for pkg in pkgs:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
# print("Done.")

# ── Verify key imports ─────────────────────────────────────────────────────────
import importlib, sys

required = {
    "dotenv":               "python-dotenv",
    "unstructured":         "unstructured[all-docs]",
    "pdfplumber":           "pdfplumber",
    "fitz":                 "pymupdf",
    "langchain":            "langchain",
    "langchain_community":  "langchain-community",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_groq":       "langchain-groq",
    "langchain_huggingface":"langchain-huggingface",
    "google.generativeai":  "google-generativeai",
    "sentence_transformers":"sentence-transformers",
    "chromadb":             "chromadb",
    "tiktoken":             "tiktoken",
    "cv2":                  "opencv-python-headless",
    "matplotlib":           "matplotlib",
    "ipywidgets":           "ipywidgets",
    "tqdm":                 "tqdm",
}

all_ok = True
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f"  ✓  {module}")
    except ImportError:
        print(f"  ✗  {module}  →  pip install {pkg}")
        all_ok = False

print()
print("All imports OK ✓" if all_ok else "⚠  Some packages missing — run the install block above.")
print(f"Python: {sys.version}")

## Step 2 — Environment Setup (API Keys)
API keys are loaded from `.env` in the project root. **Never hard-code keys in code.**

In [ ]:
import os
from dotenv import load_dotenv

# Load keys from .env (project root)
load_dotenv(dotenv_path=".env", override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY",   "")
HF_TOKEN       = os.getenv("HF_TOKEN",        "")

# Propagate to environment so LangChain adapters pick them up
os.environ["GOOGLE_API_KEY"]            = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"]             = GROQ_API_KEY
os.environ["HF_TOKEN"]                 = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

def _check(name, val):
    status = "✓ loaded" if val else "✗ MISSING — add to .env"
    print(f"  {name:<30} {status}")

print("API Key Status:")
_check("GOOGLE_API_KEY (Gemini)",  GOOGLE_API_KEY)
_check("GROQ_API_KEY",             GROQ_API_KEY)
_check("HF_TOKEN (HuggingFace)",   HF_TOKEN)

print()
if not all([GOOGLE_API_KEY, GROQ_API_KEY, HF_TOKEN]):
    print("⚠  One or more keys missing. Edit .env and re-run this cell.")
else:
    print("All API keys loaded ✓")

In [ ]:
import os
from pathlib import Path

# ── Directory constants (used throughout notebook) ──────────────────────────
CONTENT_DIR = "./content"          # PDF storage
IMAGES_DIR  = "./content/images"   # extracted image files (optional)
CHROMA_DIR  = "./chroma_db"        # persistent ChromaDB storage

for d in [CONTENT_DIR, IMAGES_DIR, CHROMA_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"  Ready: {d}")

print("\nDirectories set up ✓")

## Step 3 — Load Your PDF
Place your PDF in the `content/` folder, or use the helper below to copy it from any path.
You can also run the **widget uploader** for drag-and-drop upload inside Jupyter.

In [ ]:
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Option A: copy PDF from any path on your machine ───────────────────────
def load_pdf_from_path(source_path: str, rename: str = "paper.pdf") -> str:
    """
    Copy a PDF to content/ for processing.
    Usage: PDF_PATH = load_pdf_from_path(r"C:\\Users\\You\\Downloads\\paper.pdf")
    """
    src = Path(source_path)
    if not src.exists():
        raise FileNotFoundError(f"File not found: {source_path}")
    dest = Path(CONTENT_DIR) / rename
    shutil.copy2(src, dest)
    print(f"PDF copied → {dest}")
    return str(dest)

# ── Option B: drag-and-drop uploader widget ────────────────────────────────
def pdf_uploader_widget():
    """
    Interactive widget: drag a PDF onto the upload button.
    Returns a dict {'path': ...} that updates when upload is done.
    """
    uploader = widgets.FileUpload(accept=".pdf", multiple=False,
                                   description="Upload PDF",
                                   button_style="primary")
    status   = widgets.Output()
    state    = {"path": None}

    def _on_upload(change):
        with status:
            clear_output()
            for fname, fdata in uploader.value.items():
                dest = Path(CONTENT_DIR) / fname
                dest.write_bytes(fdata["content"])
                state["path"] = str(dest)
                print(f"✓ Uploaded: {dest}  ({len(fdata['content'])//1024} KB)")

    uploader.observe(_on_upload, names="value")
    display(widgets.VBox([
        widgets.HTML("<b>Upload a PDF to analyse:</b>"),
        uploader, status
    ]))
    return state   # access state["path"] after uploading

# ── Set PDF_PATH ────────────────────────────────────────────────────────────
#  → Option A (path copy):
#     PDF_PATH = load_pdf_from_path(r"C:\path\to\your.pdf")
#
#  → Option B (widget upload):
#     upload_state = pdf_uploader_widget()
#     # after uploading: PDF_PATH = upload_state["path"]
#
#  → Option C (already in content/):
PDF_PATH = "./content/paper.pdf"

print(f"PDF_PATH set to: {PDF_PATH}")
if not Path(PDF_PATH).exists():
    print()
    print("⚠  File not found at that path yet.")
    print("   Use load_pdf_from_path() or the upload widget, OR copy your PDF to ./content/paper.pdf")
else:
    size_kb = Path(PDF_PATH).stat().st_size // 1024
    print(f"   File exists ✓  ({size_kb} KB)")

## Step 4 — Extract All Multimodal Elements from PDF
Uses **Unstructured** with `hi_res` strategy (powered by a document layout model) to detect and extract:
- Text blocks (paragraphs, headers, captions)
- Tables (with HTML representation)
- Images / Figures / Charts / Diagrams (as base64)
- Equations and form elements are extracted as text where possible

> **Note:** `hi_res` strategy is thorough but slow. For a 20-page paper: ~1-3 min.

In [ ]:
from unstructured.partition.pdf import partition_pdf
from pathlib import Path

def extract_pdf_elements(pdf_path: str) -> list:
    """
    Partition a PDF into multimodal elements using Unstructured hi_res.

    Returns a list of chunks (CompositeElement, Table, etc.).
    Each CompositeElement may contain embedded Image elements in
    chunk.metadata.orig_elements.
    """
    if not Path(pdf_path).exists():
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}\n"
            "Set PDF_PATH correctly in Step 3 and re-run."
        )

    print(f"Partitioning: {pdf_path}")
    print("Strategy: hi_res  (layout model + OCR)  — this may take a few minutes …\n")

    chunks = partition_pdf(
        filename=pdf_path,
        infer_table_structure=True,            # extract table structure as HTML
        strategy="hi_res",                     # best quality; requires poppler + tesseract
        extract_image_block_types=["Image"],   # pull figures/charts as base64
        extract_image_block_to_payload=True,   # embed base64 in metadata (no disk writes needed)
        chunking_strategy="by_title",          # group paragraphs under their section heading
        max_characters=10000,
        combine_text_under_n_chars=2000,
        new_after_n_chars=6000,
    )

    types = {}
    for c in chunks:
        t = type(c).__name__
        types[t] = types.get(t, 0) + 1

    print(f"Extraction complete. {len(chunks)} chunks found:")
    for t, n in sorted(types.items()):
        print(f"  {t}: {n}")

    return chunks

# ── Run extraction ──────────────────────────────────────────────────────────
chunks = extract_pdf_elements(PDF_PATH)

In [ ]:
def categorize_elements(chunks: list) -> dict:
    """
    Separate chunks into texts, tables, and images (base64 strings).

    - texts  : CompositeElement objects (paragraphs + embedded structure)
    - tables : Table objects (have .metadata.text_as_html)
    - images : list of base64 strings extracted from CompositeElement orig_elements
    """
    texts  = []
    tables = []
    images = []

    for chunk in chunks:
        ctype = type(chunk).__name__

        if "Table" in ctype:
            tables.append(chunk)

        elif "CompositeElement" in ctype:
            texts.append(chunk)

            # harvest any images embedded inside this composite chunk
            orig = getattr(getattr(chunk, "metadata", None), "orig_elements", None)
            if orig:
                for el in orig:
                    if "Image" in type(el).__name__:
                        b64 = getattr(getattr(el, "metadata", None), "image_base64", None)
                        if b64:
                            images.append(b64)

    print("Element breakdown:")
    print(f"  Text chunks : {len(texts)}")
    print(f"  Tables      : {len(tables)}")
    print(f"  Images      : {len(images)}")
    return {"texts": texts, "tables": tables, "images": images}

elements = categorize_elements(chunks)
texts    = elements["texts"]
tables   = elements["tables"]
images   = elements["images"]

## Step 5 — Inspect Extracted Elements
Preview each modality to confirm extraction quality before summarization.

In [ ]:
import base64
from IPython.display import display, HTML, Image as IPImage

# ────────────────────────────────────────────────────────────────────────────
# Shared display helpers (used throughout the notebook)
# ────────────────────────────────────────────────────────────────────────────

def display_image_b64(b64_str: str, caption: str = "", max_width: int = 700):
    """Render a base64-encoded image inline in Jupyter."""
    display(HTML(f"""
    <div style="margin:12px 0; text-align:center;">
      <img src="data:image/jpeg;base64,{b64_str}"
           style="max-width:{max_width}px; border:1px solid #ccc; border-radius:6px; box-shadow:0 2px 6px rgba(0,0,0,.15);"/>
      <p style="font-style:italic; color:#555; margin:4px 0;">{caption}</p>
    </div>
    """))

def display_table_chunk(table_chunk, index: int = 0):
    """Render a Table element as an HTML table in Jupyter."""
    page = getattr(getattr(table_chunk, "metadata", None), "page_number", "?")
    html = getattr(getattr(table_chunk, "metadata", None), "text_as_html", "")
    if not html:
        html = f"<pre>{table_chunk.text[:500]}</pre>"
    display(HTML(f"""
    <details open>
      <summary style="font-weight:bold; cursor:pointer;">
        Table {index + 1} &nbsp;|&nbsp; Page {page}
      </summary>
      <div style="overflow-x:auto; margin:8px 0; padding:4px;">
        {html}
      </div>
    </details>
    """))

def display_text_chunk(text_chunk, index: int = 0, max_chars: int = 600):
    """Render a text chunk as a styled card."""
    page = getattr(getattr(text_chunk, "metadata", None), "page_number", "?")
    preview = text_chunk.text[:max_chars].replace("<", "&lt;").replace(">", "&gt;")
    if len(text_chunk.text) > max_chars:
        preview += " …"
    display(HTML(f"""
    <details>
      <summary style="font-weight:bold; cursor:pointer;">
        Text Chunk {index + 1} &nbsp;|&nbsp; Page {page}
      </summary>
      <pre style="background:#f7f7f7; padding:10px; border-left:3px solid #007acc;
                  white-space:pre-wrap; font-size:13px;">{preview}</pre>
    </details>
    """))

print("Display helpers defined ✓")

In [ ]:
# ── Preview images ──────────────────────────────────────────────────────────
MAX_PREVIEW = 6

print(f"Showing up to {MAX_PREVIEW} of {len(images)} extracted images:\n")
if images:
    for i, img_b64 in enumerate(images[:MAX_PREVIEW]):
        display_image_b64(img_b64, caption=f"Figure {i + 1}")

In [ ]:
# ── Preview tables ──────────────────────────────────────────────────────────
print(f"Showing up to 4 of {len(tables)} extracted tables:\n")
if tables:
    for i, t in enumerate(tables[:4]):
        display_table_chunk(t, i)
else:
    print("No tables found in this document.")

In [ ]:
# ── Preview text chunks ─────────────────────────────────────────────────────
print(f"Showing up to 4 of {len(texts)} text chunks:\n")
for i, t in enumerate(texts[:4]):
    display_text_chunk(t, i)

## Step 6 — Summarize Text & Tables with Groq (Llama-3.1)
- Fast, free, open-source LLM inference
- Text chunks → concise paragraph summaries
- Tables → analytical descriptions (headers, values, trends, conclusions)

These summaries are what get embedded and searched — the **original** elements are stored separately and retrieved on match.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Groq LLM ────────────────────────────────────────────────────────────────
# Models available on Groq (free tier): llama-3.1-8b-instant, llama3-70b-8192,
#   mixtral-8x7b-32768, gemma2-9b-it
groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",   # fast + capable; swap to llama3-70b-8192 for higher quality
    temperature=0.2,
    api_key=GROQ_API_KEY,
)

# ── Text summarization prompt ────────────────────────────────────────────────
TEXT_PROMPT = ChatPromptTemplate.from_template("""
You are an expert academic assistant. Read the following passage from a research paper or document.

Write a concise, information-dense summary that captures:
- Main topic / argument
- Key facts, findings, methods, or claims
- Any named entities (authors, datasets, models, metrics)
- Do NOT start with "Here is a summary" or similar phrases

Passage:
{element}
""")

# ── Table summarization prompt ───────────────────────────────────────────────
TABLE_PROMPT = ChatPromptTemplate.from_template("""
You are an expert data analyst. Analyse the following HTML table from a research paper.

Provide a structured description covering:
- What the table measures / reports
- Column and row headers
- Key numerical values, rankings, or comparisons
- Notable trends, patterns, or conclusions
- Do NOT start with "Here is a summary" or similar phrases

Table (HTML):
{element}
""")

text_chain  = {"element": lambda x: x} | TEXT_PROMPT  | groq_llm | StrOutputParser()
table_chain = {"element": lambda x: x} | TABLE_PROMPT | groq_llm | StrOutputParser()

print("Groq summarization chains ready ✓")
print(f"  Text chunks to summarize : {len(texts)}")
print(f"  Tables to summarize      : {len(tables)}")

In [ ]:
from tqdm.notebook import tqdm

# ── Summarize text chunks ────────────────────────────────────────────────────
print("Summarizing text chunks …")
text_summaries = text_chain.batch(
    [t.text for t in texts],
    config={"max_concurrency": 4},
)
print(f"  Done: {len(text_summaries)} text summaries\n")

# ── Summarize tables ─────────────────────────────────────────────────────────
if tables:
    print("Summarizing tables …")
    tables_html     = [getattr(getattr(t, "metadata", None), "text_as_html", t.text) for t in tables]
    table_summaries = table_chain.batch(
        tables_html,
        config={"max_concurrency": 4},
    )
    print(f"  Done: {len(table_summaries)} table summaries\n")
else:
    table_summaries = []
    print("No tables to summarize.")

# ── Quick sanity check ───────────────────────────────────────────────────────
if text_summaries:
    print("── Sample text summary (chunk 0) ──────────────────────────────────")
    print(text_summaries[0][:500])
if table_summaries:
    print("\n── Sample table summary (table 0) ─────────────────────────────────")
    print(table_summaries[0][:500])

## Step 7 — Summarize Images & Figures with Gemini Vision
Gemini 1.5 Flash processes each extracted image and produces detailed descriptions covering:
- Figure type (bar chart, scatter plot, diagram, equation, photo, etc.)
- All visible text labels, axis labels, legends
- Key data, trends, and takeaways
- What the figure communicates in the context of the paper

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import time

# ── Gemini Vision model ──────────────────────────────────────────────────────
gemini_vision = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",         # fast multimodal; swap to gemini-1.5-pro for max quality
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

IMAGE_DESCRIBE_PROMPT = (
    "This image is from a research paper or PDF document. "
    "Provide a detailed and structured description:\n\n"
    "1. **Figure type**: (bar chart / line graph / scatter plot / diagram / "
    "architecture figure / equation / table screenshot / photo / other)\n"
    "2. **Visible text**: List all axis labels, titles, legends, annotations, captions\n"
    "3. **Key data / values**: Specific numbers, percentages, comparisons visible\n"
    "4. **Trends / patterns**: What the data shows\n"
    "5. **Insight**: What conclusion or point this figure supports in the paper\n\n"
    "Be specific and thorough — your description will be used for semantic search."
)

def summarize_image_gemini(b64_image: str) -> str:
    """Describe a single base64 image using Gemini Vision."""
    try:
        msg = HumanMessage(content=[
            {"type": "text", "text": IMAGE_DESCRIBE_PROMPT},
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{b64_image}"}},
        ])
        resp = gemini_vision.invoke([msg])
        return resp.content
    except Exception as exc:
        return f"[Error summarizing image: {exc}]"

# ── Run image summarization ──────────────────────────────────────────────────
if images:
    print(f"Summarizing {len(images)} images with Gemini Vision …")
    image_summaries = []
    for i, img_b64 in enumerate(images):
        print(f"  Image {i+1}/{len(images)} …", end="\r")
        summary = summarize_image_gemini(img_b64)
        image_summaries.append(summary)
        time.sleep(0.3)   # small delay to stay within free-tier rate limits
    print(f"\nDone: {len(image_summaries)} image summaries ✓")
    print("\n── Sample image summary (figure 0) ────────────────────────────────")
    print(image_summaries[0][:600])
else:
    image_summaries = []
    print("No images found in this document.")

## Step 8 — Embeddings + ChromaDB Multi-Vector Retriever
**Strategy:**
- Each modality's **summary** is embedded and stored in ChromaDB (the search layer)
- The **original** element (raw text, table HTML, image base64) is stored in a docstore keyed by the same UUID
- On retrieval: ChromaDB finds the best-matching summaries → their UUIDs fetch the originals from the docstore
- The final answer generation sees the **original** content (not just summaries)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# ── HuggingFace Embeddings ───────────────────────────────────────────────────
# all-MiniLM-L6-v2 : fast (384-dim), strong semantic similarity, runs on CPU
# Alternative       : "BAAI/bge-base-en-v1.5"  (768-dim, stronger)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},            # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True},
)

# Sanity check
sample_vec = embedding_model.embed_query("multimodal retrieval augmented generation")
print(f"Embedding model loaded ✓")
print(f"  Model : sentence-transformers/all-MiniLM-L6-v2")
print(f"  Dim   : {len(sample_vec)}")

In [ ]:
import uuid
from langchain_community.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document

DOC_ID_KEY = "doc_id"    # metadata key linking summary → original

# ── ChromaDB (persistent) ────────────────────────────────────────────────────
vectorstore = Chroma(
    collection_name="multimodal_rag",
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,          # saves to ./chroma_db automatically
)

# ── In-memory docstore for original elements ─────────────────────────────────
# Keys   : UUID strings
# Values : original elements (Unstructured chunk, HTML string, base64 image string)
docstore = InMemoryStore()

# ── MultiVectorRetriever ─────────────────────────────────────────────────────
# On .invoke(query):
#   1. Embeds the query
#   2. Finds top-k summaries in ChromaDB
#   3. Looks up their UUIDs in docstore
#   4. Returns original elements
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    id_key=DOC_ID_KEY,
    search_kwargs={"k": 6},       # retrieve top 6 most relevant chunks
)

print("ChromaDB + MultiVectorRetriever initialized ✓")
print(f"  Persist directory : {CHROMA_DIR}")
print(f"  Search k          : 6")

In [ ]:
def _add_to_retriever(
    originals: list,
    summaries: list,
    retriever: MultiVectorRetriever,
    modality: str,
    extra_meta_fn=None,
) -> list:
    """
    Generic indexer: embeds summaries → ChromaDB, stores originals → docstore.

    Args:
        originals    : list of original elements to store (chunks, html, b64)
        summaries    : list of text summaries (same length as originals)
        retriever    : MultiVectorRetriever
        modality     : "text" | "table" | "image"
        extra_meta_fn: optional callable(i, original) → dict of extra metadata

    Returns list of generated UUIDs.
    """
    if not originals:
        return []

    ids = [str(uuid.uuid4()) for _ in originals]

    summary_docs = []
    for i, (doc_id, summary) in enumerate(zip(ids, summaries)):
        meta = {DOC_ID_KEY: doc_id, "modality": modality}
        if extra_meta_fn:
            meta.update(extra_meta_fn(i, originals[i]))
        summary_docs.append(Document(page_content=summary, metadata=meta))

    retriever.vectorstore.add_documents(summary_docs)
    retriever.docstore.mset(list(zip(ids, originals)))
    return ids


# ── Index texts ──────────────────────────────────────────────────────────────
print("Indexing text chunks …")
text_ids = _add_to_retriever(
    texts, text_summaries, retriever, modality="text",
    extra_meta_fn=lambda i, el: {
        "page": getattr(getattr(el, "metadata", None), "page_number", -1),
    },
)

# ── Index tables ─────────────────────────────────────────────────────────────
print("Indexing tables …")
table_ids = _add_to_retriever(
    tables, table_summaries, retriever, modality="table",
    extra_meta_fn=lambda i, el: {
        "page": getattr(getattr(el, "metadata", None), "page_number", -1),
        "html": getattr(getattr(el, "metadata", None), "text_as_html", ""),
    },
)

# ── Index images ─────────────────────────────────────────────────────────────
print("Indexing images …")
# For images, the 'original' stored in docstore is the raw base64 string
image_ids = _add_to_retriever(
    images, image_summaries, retriever, modality="image",
)

# ── Summary ──────────────────────────────────────────────────────────────────
total = len(text_ids) + len(table_ids) + len(image_ids)
print(f"\nIndexing complete ✓")
print(f"  Text chunks : {len(text_ids)}")
print(f"  Tables      : {len(table_ids)}")
print(f"  Images      : {len(image_ids)}")
print(f"  Total       : {total}")
print(f"  ChromaDB    : {vectorstore._collection.count()} vectors stored")

In [ ]:
# ── Verify retrieval works ───────────────────────────────────────────────────
test_q = "What is the main topic and contribution of this paper?"
print(f"Test query: '{test_q}'\n")

raw = retriever.invoke(test_q)
print(f"Retrieved {len(raw)} docs:\n")

for i, doc in enumerate(raw):
    dtype = type(doc).__name__
    if isinstance(doc, str):
        # base64 image
        print(f"  [{i+1}] IMAGE  (base64 length={len(doc)})")
    elif hasattr(doc, "metadata") and "Table" in dtype:
        page = getattr(getattr(doc, "metadata", None), "page_number", "?")
        print(f"  [{i+1}] TABLE  (page {page}): {doc.text[:120]} …")
    elif hasattr(doc, "text"):
        page = getattr(getattr(doc, "metadata", None), "page_number", "?")
        print(f"  [{i+1}] TEXT   (page {page}): {doc.text[:120]} …")
    else:
        print(f"  [{i+1}] {dtype}: {str(doc)[:120]}")

## Step 9 — RAG Pipeline with Source Citation
The pipeline:
1. Embeds the user query
2. Retrieves top-k original elements (text + tables + images)
3. Builds a multimodal prompt (text context + images attached)
4. Sends to **Gemini 1.5 Flash** for final answer generation
5. Returns both the answer **and** all source elements used

In [ ]:
from base64 import b64decode

def classify_docs(raw_docs: list) -> dict:
    """
    Split raw retrieved docs into three modality buckets.
    - texts  : Unstructured CompositeElement objects
    - tables : Unstructured Table objects
    - images : raw base64 strings
    """
    out = {"texts": [], "tables": [], "images": []}
    for doc in raw_docs:
        dtype = type(doc).__name__
        if isinstance(doc, str):
            # Images stored as bare base64 strings
            try:
                b64decode(doc[:100])          # quick validity check
                out["images"].append(doc)
            except Exception:
                out["texts"].append(doc)      # treat undecodable strings as text
        elif "Table" in dtype:
            out["tables"].append(doc)
        else:
            out["texts"].append(doc)
    return out


def build_rag_prompt(context: dict, question: str) -> list:
    """
    Build a HumanMessage list that passes text + tables as context string
    and attaches images for Gemini's vision capability.
    """
    parts = []

    # ── Text context ─────────────────────────────────────────────────────────
    for chunk in context["texts"]:
        content = chunk.text if hasattr(chunk, "text") else str(chunk)
        page    = getattr(getattr(chunk, "metadata", None), "page_number", "?")
        parts.append(f"[TEXT — page {page}]\n{content}")

    # ── Table context ─────────────────────────────────────────────────────────
    for tbl in context["tables"]:
        html = getattr(getattr(tbl, "metadata", None), "text_as_html", tbl.text)
        page = getattr(getattr(tbl, "metadata", None), "page_number", "?")
        parts.append(f"[TABLE — page {page}]\n{html}")

    ctx_str = "\n\n---\n\n".join(parts) if parts else "(no text or table context retrieved)"

    system_instruction = (
        "You are an expert research assistant. "
        "Answer the question using ONLY the provided context (text, tables, images). "
        "Rules:\n"
        "- Cite page numbers whenever you reference text or tables, e.g. (page 4)\n"
        "- If you reference a figure or chart, say what you see in it\n"
        "- If the answer involves numbers from a table, quote the exact values\n"
        "- If the context is insufficient, say so clearly\n"
        "- Be thorough, specific, and well-structured\n\n"
        f"CONTEXT:\n{ctx_str}\n\n"
        f"QUESTION: {question}"
    )

    content = [{"type": "text", "text": system_instruction}]

    # ── Attach images ─────────────────────────────────────────────────────────
    for img_b64 in context["images"]:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
        })

    return [HumanMessage(content=content)]


# ── Gemini for final answer generation ──────────────────────────────────────
gemini_answer = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1,
)

print("RAG pipeline helpers defined ✓")

In [ ]:
from IPython.display import display, HTML, Markdown

def display_sources(context: dict):
    """Render all retrieved source elements (text, tables, images) inline."""

    # ── Text sources ──────────────────────────────────────────────────────────
    if context["texts"]:
        display(HTML(
            f"<hr/><h4 style='color:#007acc;'>📄 Source Text "
            f"({len(context['texts'])} chunk(s) retrieved)</h4>"
        ))
        for i, chunk in enumerate(context["texts"]):
            content = chunk.text if hasattr(chunk, "text") else str(chunk)
            page    = getattr(getattr(chunk, "metadata", None), "page_number", "?")
            safe    = content[:1200].replace("<", "&lt;").replace(">", "&gt;")
            if len(content) > 1200:
                safe += "\n… (truncated)"
            display(HTML(f"""
            <details style="margin:6px 0;">
              <summary style="cursor:pointer; font-weight:bold;">
                Text Chunk {i+1} — Page {page}
              </summary>
              <pre style="background:#f7f7f7; padding:10px; border-left:4px solid #007acc;
                          white-space:pre-wrap; font-size:12.5px; margin-top:4px;">{safe}</pre>
            </details>
            """))

    # ── Table sources ─────────────────────────────────────────────────────────
    if context["tables"]:
        display(HTML(
            f"<hr/><h4 style='color:#2e7d32;'>📊 Source Tables "
            f"({len(context['tables'])} table(s) retrieved)</h4>"
        ))
        for i, tbl in enumerate(context["tables"]):
            html = getattr(getattr(tbl, "metadata", None), "text_as_html", "")
            page = getattr(getattr(tbl, "metadata", None), "page_number", "?")
            if not html:
                html = f"<pre>{tbl.text[:500]}</pre>"
            display(HTML(f"""
            <details open style="margin:6px 0;">
              <summary style="cursor:pointer; font-weight:bold;">
                Table {i+1} — Page {page}
              </summary>
              <div style="overflow-x:auto; margin:8px 0; padding:4px; border-left:4px solid #2e7d32;">
                {html}
              </div>
            </details>
            """))

    # ── Image sources ─────────────────────────────────────────────────────────
    if context["images"]:
        display(HTML(
            f"<hr/><h4 style='color:#6a1b9a;'>🖼 Source Figures / Images "
            f"({len(context['images'])} image(s) retrieved)</h4>"
        ))
        for i, img_b64 in enumerate(context["images"]):
            display(HTML(f"""
            <details open style="margin:6px 0;">
              <summary style="cursor:pointer; font-weight:bold;">Figure {i+1}</summary>
              <div style="margin:10px; text-align:center;">
                <img src="data:image/jpeg;base64,{img_b64}"
                     style="max-width:720px; border:1px solid #ccc; border-radius:6px;
                            box-shadow:0 2px 8px rgba(0,0,0,.2);"/>
              </div>
            </details>
            """))

    display(HTML("<hr/>"))


def query_paper(question: str, show_sources: bool = True) -> dict:
    """
    Main entry point: ask any question about the loaded document.

    Args:
        question     : natural-language question
        show_sources : if True, display all retrieved source elements inline

    Returns:
        dict with keys: answer (str), context (dict)
    """
    # 1. Retrieve
    raw_docs = retriever.invoke(question)
    context  = classify_docs(raw_docs)

    # 2. Generate answer
    messages = build_rag_prompt(context, question)
    response = gemini_answer.invoke(messages)
    answer   = response.content

    # 3. Display
    display(HTML(f"""
    <div style="background:#e8f4fd; border:1px solid #007acc; border-radius:8px;
                padding:16px 20px; margin:12px 0;">
      <h3 style="color:#007acc; margin-top:0;">Question</h3>
      <p style="font-size:15px;">{question}</p>
    </div>
    """))
    display(HTML("<h3>Answer</h3>"))
    display(Markdown(answer))

    if show_sources:
        display(HTML("<h3>Sources Used</h3>"))
        display_sources(context)

    return {"answer": answer, "context": context}

print("query_paper() function ready ✓")

## Step 10 — Ask Questions About Your Document
Use `query_paper("your question")` for any question.
The answer will be followed by **all source elements** (text, tables, images) that contributed.

In [ ]:
# ── Example question 1: Overview ────────────────────────────────────────────
result = query_paper("What is the main topic and key contribution of this paper?")

In [ ]:
# ── Example question 2: Tables / numbers ────────────────────────────────────
result2 = query_paper(
    "What experimental results or performance numbers are reported? "
    "Include any benchmark comparisons from tables."
)

In [ ]:
# ── Example question 3: Figures / images ────────────────────────────────────
result3 = query_paper(
    "Describe the figures and diagrams in this document. "
    "What do they show and what insights do they provide?"
)

In [ ]:
# ── Example question 4: Methods ──────────────────────────────────────────────
result4 = query_paper("What methodology or approach does the paper propose?")

In [ ]:
# ── YOUR QUESTION — edit and run this cell ───────────────────────────────────
your_question = "What are the conclusions and future work directions?"

result_custom = query_paper(your_question)

## Step 11 — Interactive Q&A Widget
A live input box — type any question and click **Ask**. No need to edit code cells.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def launch_qa_widget():
    """
    Interactive Q&A panel for the loaded document.
    Type your question, press Ask or Shift+Enter.
    """
    # ── Suggested questions ──────────────────────────────────────────────────
    suggestions = [
        "What is this paper about?",
        "What datasets were used?",
        "What are the key results?",
        "Describe the architecture or method",
        "What do the figures show?",
        "What are the limitations?",
        "Who are the authors?",
    ]

    style  = {"description_width": "initial"}
    layout = widgets.Layout

    question_box = widgets.Textarea(
        placeholder="Type your question here …",
        layout=layout(width="100%", height="70px"),
    )
    ask_btn   = widgets.Button(description=" Ask", button_style="primary", icon="search",
                               layout=layout(width="110px", height="36px"))
    clear_btn = widgets.Button(description="Clear", button_style="warning", icon="trash",
                               layout=layout(width="90px", height="36px"))
    suggest_dd = widgets.Dropdown(
        options=["(pick a suggestion)"] + suggestions,
        description="Suggestions:",
        style=style, layout=layout(width="60%"),
    )
    sources_toggle = widgets.Checkbox(value=True, description="Show sources",
                                       layout=layout(width="140px"))
    status_label = widgets.Label(value="")
    output       = widgets.Output()

    def _on_suggest(change):
        if change["new"] != "(pick a suggestion)":
            question_box.value = change["new"]
            suggest_dd.value   = "(pick a suggestion)"

    def _on_ask(_):
        q = question_box.value.strip()
        if not q:
            status_label.value = "⚠ Please enter a question."
            return
        status_label.value = "⏳ Searching and generating answer …"
        with output:
            clear_output(wait=True)
            try:
                query_paper(q, show_sources=sources_toggle.value)
            except Exception as exc:
                display(HTML(f"<b style='color:red;'>Error: {exc}</b>"))
        status_label.value = "✓ Done"

    def _on_clear(_):
        question_box.value = ""
        status_label.value = ""
        with output:
            clear_output()

    suggest_dd.observe(_on_suggest, names="value")
    ask_btn.on_click(_on_ask)
    clear_btn.on_click(_on_clear)

    display(widgets.VBox([
        widgets.HTML("<h3 style='margin-bottom:6px;'>📖 Document Q&A</h3>"),
        suggest_dd,
        question_box,
        widgets.HBox([ask_btn, clear_btn, sources_toggle, status_label]),
        widgets.HTML("<hr/>"),
        output,
    ]))

launch_qa_widget()

## Step 12 — Document Statistics Dashboard
Summary of what was extracted and indexed from the document.

In [ ]:
from IPython.display import display, HTML
from pathlib import Path

def show_document_stats():
    """Render a summary dashboard of what was processed and indexed."""
    pdf_name = Path(PDF_PATH).name if "PDF_PATH" in dir() else "—"
    total    = len(texts) + len(tables) + len(images)
    chroma_n = vectorstore._collection.count()

    display(HTML(f"""
    <div style="background:#f0f7ff; border:2px solid #007acc; border-radius:10px;
                padding:20px 28px; margin:14px 0; font-family:sans-serif;">
      <h2 style="color:#007acc; margin-top:0;">Document Processing Report</h2>

      <table style="width:100%; border-collapse:collapse; font-size:14px;">
        <tr style="background:#007acc; color:white;">
          <th style="padding:8px 12px; text-align:left;">Item</th>
          <th style="padding:8px 12px; text-align:center;">Count</th>
          <th style="padding:8px 12px; text-align:left;">Model Used</th>
        </tr>
        <tr style="background:#fff;">
          <td style="padding:8px 12px;">PDF loaded</td>
          <td style="padding:8px 12px; text-align:center;">{pdf_name}</td>
          <td style="padding:8px 12px;">Unstructured (hi_res)</td>
        </tr>
        <tr style="background:#e8f4fd;">
          <td style="padding:8px 12px;">Text chunks extracted</td>
          <td style="padding:8px 12px; text-align:center;"><b>{len(texts)}</b></td>
          <td style="padding:8px 12px;">Groq / Llama-3.1-8b-instant</td>
        </tr>
        <tr style="background:#fff;">
          <td style="padding:8px 12px;">Tables extracted</td>
          <td style="padding:8px 12px; text-align:center;"><b>{len(tables)}</b></td>
          <td style="padding:8px 12px;">Groq / Llama-3.1-8b-instant</td>
        </tr>
        <tr style="background:#e8f4fd;">
          <td style="padding:8px 12px;">Images / Figures extracted</td>
          <td style="padding:8px 12px; text-align:center;"><b>{len(images)}</b></td>
          <td style="padding:8px 12px;">Gemini 1.5 Flash (Vision)</td>
        </tr>
        <tr style="background:#fff;">
          <td style="padding:8px 12px;">Total elements indexed</td>
          <td style="padding:8px 12px; text-align:center;"><b>{total}</b></td>
          <td style="padding:8px 12px;">HuggingFace all-MiniLM-L6-v2</td>
        </tr>
        <tr style="background:#e8f4fd;">
          <td style="padding:8px 12px;">ChromaDB vectors stored</td>
          <td style="padding:8px 12px; text-align:center;"><b>{chroma_n}</b></td>
          <td style="padding:8px 12px;">ChromaDB ({CHROMA_DIR})</td>
        </tr>
        <tr style="background:#fff;">
          <td style="padding:8px 12px;">Answer generation</td>
          <td style="padding:8px 12px; text-align:center;">—</td>
          <td style="padding:8px 12px;">Gemini 1.5 Flash (multimodal)</td>
        </tr>
      </table>

      <p style="margin-top:14px; font-size:13px; color:#555;">
        <b>Multi-vector retrieval:</b> summaries are embedded for search;
        original elements (text, HTML tables, base64 images) are stored separately
        and returned for grounding the LLM answer.
      </p>
    </div>
    """))

show_document_stats()

## Step 13 — Persist & Reload ChromaDB
On subsequent runs you can **skip Steps 4-8** and reload the existing index directly.
This saves time for large documents that have already been processed.

In [ ]:
# ── Persist: ChromaDB auto-persists with persist_directory ──────────────────
# Nothing extra needed for chromadb >= 0.4. Data is on disk at CHROMA_DIR.
print(f"ChromaDB persisted at: {CHROMA_DIR}")
print(f"Vectors in collection: {vectorstore._collection.count()}")
print()
print("To reload on a future run without re-processing the PDF:")
print("""
  from langchain_community.vectorstores import Chroma
  from langchain.storage import InMemoryStore
  from langchain.retrievers.multi_vector import MultiVectorRetriever

  vectorstore = Chroma(
      collection_name="multimodal_rag",
      embedding_function=embedding_model,
      persist_directory="./chroma_db",
  )
  docstore  = InMemoryStore()       # note: in-memory store resets on restart
  retriever = MultiVectorRetriever(
      vectorstore=vectorstore,
      docstore=docstore,
      id_key="doc_id",
      search_kwargs={"k": 6},
  )
  # Then call query_paper("your question") — text/table/image originals
  # will be re-fetched from ChromaDB metadata if needed.
""")

## Step 14 — Advanced: PyMuPDF Deep Metadata Extraction (Supplemental)
`PyMuPDF (fitz)` provides low-level access to PDF metadata, fonts, page dimensions, hyperlinks, annotations, and embedded objects. Use this for any metadata-level questions.

In [ ]:
import fitz   # PyMuPDF

def extract_pdf_metadata(pdf_path: str) -> dict:
    """
    Extract document-level metadata, TOC, annotations, and links using PyMuPDF.
    """
    doc = fitz.open(pdf_path)

    meta = doc.metadata
    toc  = doc.get_toc()       # [[level, title, page], ...]

    page_info = []
    annotations = []
    links_all   = []

    for page_num, page in enumerate(doc):
        info = {
            "page"  : page_num + 1,
            "width" : round(page.rect.width,  1),
            "height": round(page.rect.height, 1),
            "blocks": len(page.get_text("blocks")),
        }
        page_info.append(info)

        for annot in page.annots():
            annotations.append({
                "page": page_num + 1,
                "type": annot.type[1],
                "content": annot.info.get("content", ""),
            })

        for link in page.get_links():
            if link.get("uri"):
                links_all.append({"page": page_num + 1, "url": link["uri"]})

    doc.close()
    return {
        "metadata"   : meta,
        "num_pages"  : len(page_info),
        "toc"        : toc,
        "pages"      : page_info,
        "annotations": annotations,
        "links"      : links_all,
    }


if Path(PDF_PATH).exists():
    pdf_meta = extract_pdf_metadata(PDF_PATH)

    print("── PDF Metadata ────────────────────────────────────────────────────")
    for k, v in pdf_meta["metadata"].items():
        if v:
            print(f"  {k:<14}: {v}")

    print(f"\n── Pages: {pdf_meta['num_pages']}")
    if pdf_meta["toc"]:
        print(f"\n── Table of Contents ({len(pdf_meta['toc'])} entries):")
        for level, title, page in pdf_meta["toc"][:20]:
            print(f"  {'  ' * (level-1)}{title}  (p.{page})")
    if pdf_meta["links"]:
        print(f"\n── External Links ({len(pdf_meta['links'])}):")
        for lnk in pdf_meta["links"][:10]:
            print(f"  p.{lnk['page']}: {lnk['url']}")
else:
    print("PDF_PATH not found — set it in Step 3 first.")

## Step 15 — Advanced: pdfplumber Precision Table Extraction (Supplemental)
`pdfplumber` uses heuristic line detection for tables. Useful as a **second pass** when Unstructured misses or garbles a table.

In [ ]:
import pdfplumber
import pandas as pd
from IPython.display import display, HTML

def extract_tables_pdfplumber(pdf_path: str, max_tables: int = 20) -> list:
    """
    Extract tables from PDF using pdfplumber.
    Returns list of (page_number, DataFrame) tuples.
    """
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables()
            for tbl_data in page_tables:
                if tbl_data:
                    # first row as header if it looks like a header
                    try:
                        df = pd.DataFrame(tbl_data[1:], columns=tbl_data[0])
                    except Exception:
                        df = pd.DataFrame(tbl_data)
                    results.append((page_num, df))
                    if len(results) >= max_tables:
                        break
            if len(results) >= max_tables:
                break
    return results


if Path(PDF_PATH).exists():
    plumber_tables = extract_tables_pdfplumber(PDF_PATH)
    print(f"pdfplumber found {len(plumber_tables)} tables\n")

    for i, (page, df) in enumerate(plumber_tables[:3]):
        display(HTML(f"<b>pdfplumber Table {i+1} — Page {page}</b>"))
        display(df.fillna("").head(10).style
                  .set_table_styles([{
                      "selector": "th",
                      "props": [("background","#007acc"),("color","white"),("padding","6px")]
                  }]))
        print()
else:
    plumber_tables = []
    print("PDF not found — skipping pdfplumber extraction.")

## Step 16 — Advanced: Add pdfplumber Tables to the Index
If pdfplumber found additional / better tables, summarise and index them too.

In [ ]:
if plumber_tables:
    # Convert DataFrames to HTML + summarise with Groq
    extra_table_htmls = [df.to_html(index=False) for _, df in plumber_tables]
    print(f"Summarizing {len(extra_table_htmls)} pdfplumber tables …")
    extra_table_summaries = table_chain.batch(extra_table_htmls, config={"max_concurrency": 4})

    # Store originals as HTML strings (not Unstructured objects)
    extra_ids = [str(uuid.uuid4()) for _ in extra_table_htmls]
    extra_docs = [
        Document(
            page_content=summary,
            metadata={
                DOC_ID_KEY: extra_ids[i],
                "modality": "table",
                "page": plumber_tables[i][0],
                "source": "pdfplumber",
            },
        )
        for i, summary in enumerate(extra_table_summaries)
    ]
    retriever.vectorstore.add_documents(extra_docs)
    retriever.docstore.mset(list(zip(extra_ids, extra_table_htmls)))

    print(f"Indexed {len(extra_ids)} additional pdfplumber tables ✓")
    print(f"ChromaDB now has {vectorstore._collection.count()} vectors")
else:
    print("No pdfplumber tables to add (either none found or PDF not yet loaded).")